# ITC560 Assessment 4: Greenhouse IoT Analytics

This notebook analyses a simulated smart greenhouse dataset. The readings represent IoT telemetry from two greenhouse zones, including sensors, actuators, device health, and gateway/network measurements.

Run the variant selector before completing this notebook. Use your selected dataset and commit your outputs.

The starter code uses basic Python ideas plus a small amount of `pandas` for tables and `matplotlib` for charts. You are expected to adapt the examples, add your own indicators, and explain your choices in the report.

## 1. Load Selected Dataset

Use the manifest in `data/selected/variant_manifest.json` to load your dataset.

The manifest records which dataset variant belongs to this repository. Keeping it means your analysis can be reproduced later by your marker.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

from iot_analysis import count_anomalies, identify_risk_windows, load_rows, numeric_summary
from variant_selector import MANIFEST_PATH, generate_dataset, select_variant

manifest = select_variant()
dataset_path = generate_dataset(manifest)
rows = load_rows(dataset_path)
df = pd.read_csv(dataset_path, parse_dates=['timestamp'])

manifest, df.head()

({'id': 'greenhouse-d',
  'name': 'Water-constrained greenhouse',
  'temperature_offset': 1.2,
  'humidity_offset': -8.0,
  'soil_offset': -9.0,
  'fault_window': [90, 96],
  'gateway_window': [142, 148],
  'dropout_window': [44, 46],
  'fault_zone': 'south-bed',
  'selected_at_utc': '2026-10-14T04:20:13+00:00',
  'dataset_file': 'greenhouse-d_sensor_readings.csv'},
             timestamp       zone  air_temperature_c  relative_humidity_pct  \
 0 2026-09-21 00:00:00  north-bed              20.35                  59.18   
 1 2026-09-21 00:00:00  south-bed              19.38                  62.98   
 2 2026-09-21 01:00:00  north-bed              19.77                  58.50   
 3 2026-09-21 01:00:00  south-bed              19.02                  62.66   
 4 2026-09-21 02:00:00  north-bed              19.74                  60.83   
 
    soil_moisture_pct  light_lux  co2_ppm  door_open  fan_active  \
 0              45.50      87.84    449.0          1           0   
 1              43.

## 2. Explore And Summarise

Calculate summary statistics and identify important events. Add your own analysis cells as needed.

Think about which readings are sensor evidence, which readings show actuator behaviour, which readings show device or gateway quality, and which readings may need cleaning before interpretation.

The next cell gives you a first view of the data. Use it to decide what extra exploration you need.

In [2]:
# Helper functions give a first summary without requiring complex pandas code.
summary = numeric_summary(rows)
anomaly_counts = count_anomalies(rows)
risk_windows = identify_risk_windows(rows)

# 1. Identify missing values per column
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0]

# 2. Identify physically impossible sensor readings (negative light_lux)
negative_light_count = (df['light_lux'] < 0).sum()
negative_light_rows = df[df['light_lux'] < 0][['timestamp', 'zone', 'light_lux', 'sensor_status', 'anomaly_label']]

# 3. Clean data into dedicated cleaned columns (preserving raw columns for downstream risk flags)
df['light_lux_clean'] = df['light_lux'].mask(df['light_lux'] < 0, pd.NA)
df['light_lux_clean'] = df.groupby('zone')['light_lux_clean'].transform(lambda s: s.interpolate(method='linear').bfill().ffill())
df['soil_moisture_clean'] = df.groupby('zone')['soil_moisture_pct'].transform(lambda s: s.interpolate(method='linear').bfill().ffill())

# 4. Count records and anomaly types by greenhouse zone
zone_counts = df['zone'].value_counts()
anomalies_by_zone = pd.crosstab(df['zone'], df['anomaly_label'])

print("--- DATA QUALITY & CLEANING SUMMARY ---")
print("Missing values (raw):\n", missing_values)
print(f"\nNegative light_lux readings found: {negative_light_count}")
print(negative_light_rows)
print("\n--- ANOMALIES BY ZONE ---")
print(anomalies_by_zone)

summary, anomaly_counts, len(risk_windows), missing_values, zone_counts



--- DATA QUALITY & CLEANING SUMMARY ---
Missing values (raw):
 soil_moisture_pct    3
light_lux            2
dtype: int64

Negative light_lux readings found: 14
              timestamp       zone  light_lux sensor_status      anomaly_label
5   2026-09-21 02:00:00  south-bed     -16.70            ok             normal
7   2026-09-21 03:00:00  south-bed      -3.60            ok             normal
53  2026-09-22 02:00:00  south-bed      -8.72            ok             normal
95  2026-09-22 23:00:00  south-bed      -4.53            ok             normal
99  2026-09-23 01:00:00  south-bed     -19.12            ok             normal
145 2026-09-24 00:00:00  south-bed      -2.40            ok             normal
191 2026-09-24 23:00:00  south-bed     -20.06            ok  ventilation_fault
193 2026-09-25 00:00:00  south-bed     -19.77            ok  ventilation_fault
197 2026-09-25 02:00:00  south-bed     -20.77            ok             normal
199 2026-09-25 03:00:00  south-bed     -19.18    

({'air_temperature_c': {'min': 18.99, 'max': 27.98, 'avg': 23.06},
  'relative_humidity_pct': {'min': 45.27, 'max': 63.62, 'avg': 55.12},
  'soil_moisture_pct': {'min': 27.61, 'max': 48.05, 'avg': 37.05},
  'light_lux': {'min': -22.33, 'max': 1070.1, 'avg': 435.38},
  'co2_ppm': {'min': 413.0, 'max': 585.17, 'avg': 484.27},
  'water_flow_lpm': {'min': 0.0, 'max': 6.17, 'avg': 0.85},
  'gateway_latency_ms': {'min': 68.0, 'max': 191.0, 'avg': 90.95},
  'packet_loss_pct': {'min': 0.0, 'max': 4.54, 'avg': 0.85},
  'gateway_rssi_dbm': {'min': -82.0, 'max': -58.0, 'avg': -67.08},
  'device_battery_pct': {'min': 82.93, 'max': 94.51, 'avg': 88.81}},
 {'normal': 303,
  'sensor_dropout': 5,
  'irrigation_gap': 7,
  'ventilation_fault': 7,
  'gateway_degraded': 14},
 19,
 soil_moisture_pct    3
 light_lux            2
 dtype: int64,
 zone
 north-bed    168
 south-bed    168
 Name: count, dtype: int64)

## 3. Derive IoT Indicators

Create useful operational indicators from the raw data. You may change the thresholds below if you justify the choice in your report.

Examples:

- `thermal_risk`: temperature is outside the desired growing range.
- `water_stress`: soil moisture is below the level that should trigger attention.
- `air_quality_risk`: CO2 readings may indicate poor ventilation.
- `network_degraded`: gateway latency or packet loss may affect timely monitoring/control.
- `device_health_risk`: battery, signal, or missing sensor readings may affect trust in the data.
- `edge_action_priority`: a local fog/edge action may be needed because environment and network symptoms overlap.

In [3]:
thresholds = {
    'temperature_high_c': 27.0,
    'temperature_low_c': 18.0,
    'soil_low_pct': 42.0,
    'soil_critical_pct': 35.0,
    'co2_high_ppm': 560.0,
    'latency_high_ms': 140.0,
    'packet_loss_high_pct': 2.0,
    'battery_low_pct': 82.0,
    'rssi_weak_dbm': -72.0,
}

# Starter derived indicators
too_hot = df['air_temperature_c'] > thresholds['temperature_high_c']
too_cold = df['air_temperature_c'] < thresholds['temperature_low_c']
df['thermal_risk'] = too_hot | too_cold

missing_soil = df['soil_moisture_pct'].isna()
soil_too_low = df['soil_moisture_pct'] < thresholds['soil_low_pct']
df['water_stress'] = missing_soil | soil_too_low

df['air_quality_risk'] = df['co2_ppm'] > thresholds['co2_high_ppm']

high_latency = df['gateway_latency_ms'] > thresholds['latency_high_ms']
high_packet_loss = df['packet_loss_pct'] > thresholds['packet_loss_high_pct']
df['network_degraded'] = high_latency | high_packet_loss

low_battery = df['device_battery_pct'] < thresholds['battery_low_pct']
weak_signal = df['gateway_rssi_dbm'] < thresholds['rssi_weak_dbm']
sensor_not_ok = df['sensor_status'] != 'ok'
df['device_health_risk'] = low_battery | weak_signal | sensor_not_ok

fan_needed = df['thermal_risk'] | df['air_quality_risk']
fan_not_running = df['fan_active'] == 0
irrigation_not_running = df['irrigation_active'] == 0
df['actuator_mismatch'] = (fan_needed & fan_not_running) | (df['water_stress'] & irrigation_not_running)

environment_needs_attention = df['thermal_risk'] | df['water_stress'] | df['air_quality_risk']
remote_control_less_reliable = df['network_degraded'] | df['device_health_risk']
df['edge_action_priority'] = environment_needs_attention & remote_control_less_reliable

# COMPLETED TODO: Additional derived indicators for water-constrained greenhouse operations
df['severe_water_stress'] = (missing_soil | (df['soil_moisture_pct'] < thresholds['soil_critical_pct'])) & irrigation_not_running
df['calibration_drift_risk'] = (df['light_lux'] < 0) & (df['sensor_status'] == 'ok')

# Print complete untruncated summaries for report evidence
indicator_cols = [
    'thermal_risk', 'water_stress', 'severe_water_stress', 'air_quality_risk',
    'network_degraded', 'device_health_risk', 'calibration_drift_risk',
    'actuator_mismatch', 'edge_action_priority'
]
print("--- DERIVED INDICATORS COUNT BY ZONE ---")
print(df.groupby('zone')[indicator_cols].sum().to_string())

print("\n--- FULL ANOMALY LABELS BY ZONE ---")
print(pd.crosstab(df['zone'], df['anomaly_label']).to_string())

print("\n--- MISSING VALUE ROWS ---")
print(df[df['soil_moisture_pct'].isna() | df['light_lux'].isna()][['timestamp', 'zone', 'soil_moisture_pct', 'light_lux', 'sensor_status', 'anomaly_label']].to_string())

df[['timestamp', 'zone', 'air_temperature_c', 'soil_moisture_pct', 'co2_ppm', 'thermal_risk', 'water_stress', 'severe_water_stress', 'calibration_drift_risk', 'actuator_mismatch', 'edge_action_priority']].head()



--- DERIVED INDICATORS COUNT BY ZONE ---
           thermal_risk  water_stress  severe_water_stress  air_quality_risk  network_degraded  device_health_risk  calibration_drift_risk  actuator_mismatch  edge_action_priority
zone                                                                                                                                                                               
north-bed            15           157                   74                 0                 7                   6                       0                142                     9
south-bed             3           129                   42                 4                 7                  48                      14                118                    36

--- FULL ANOMALY LABELS BY ZONE ---
anomaly_label  gateway_degraded  irrigation_gap  normal  sensor_dropout  ventilation_fault
zone                                                                                      
north-bed           

,timestamp,zone,air_temperature_c,soil_moisture_pct,co2_ppm,thermal_risk,water_stress,severe_water_stress,calibration_drift_risk,actuator_mismatch,edge_action_priority
0,2026-09-21 00:00:00,north-bed,20.35,45.50,449.0,False,False,False,False,False,False
1,2026-09-21 00:00:00,south-bed,19.38,43.88,438.0,False,False,False,False,False,False
2,2026-09-21 01:00:00,north-bed,19.77,41.31,454.0,False,True,False,False,True,False
3,2026-09-21 01:00:00,south-bed,19.02,44.05,413.0,False,False,False,False,False,False
4,2026-09-21 02:00:00,north-bed,19.74,39.90,452.0,False,True,False,False,True,False


## 4. Visualise And Select Evidence

Create charts or tables that support your report. Save important outputs into `outputs/`. Your final repository should contain at least four useful output files that you refer to in your report, including at least one output you create or materially adapt beyond the starter examples.

Choose evidence that helps answer an IoT question, such as when local control is needed, whether actuator behaviour matches sensor conditions, or whether network quality affects monitoring.

In [4]:

outputs_dir = ROOT / 'outputs'
outputs_dir.mkdir(exist_ok=True)

# 1. Output Table 1: Rows needing attention (adapted with custom derived indicators)
risk_condition = (
    df['thermal_risk'] | df['water_stress'] | df['severe_water_stress'] |
    df['air_quality_risk'] | df['network_degraded'] | df['device_health_risk'] |
    df['calibration_drift_risk']
)

risk_columns = [
    'timestamp', 'zone', 'air_temperature_c', 'relative_humidity_pct', 'soil_moisture_pct',
    'light_lux', 'co2_ppm', 'gateway_latency_ms', 'packet_loss_pct', 'gateway_rssi_dbm',
    'device_battery_pct', 'fan_active', 'irrigation_active', 'thermal_risk',
    'water_stress', 'severe_water_stress', 'air_quality_risk', 'network_degraded',
    'device_health_risk', 'calibration_drift_risk', 'actuator_mismatch',
    'edge_action_priority', 'anomaly_label'
]

risk_table = df.loc[risk_condition, risk_columns]
risk_table.to_csv(outputs_dir / 'greenhouse_risk_windows.csv', index=False)

# 2. Output Table 2: Zone summary table (materially adapted with full operational metrics)
zone_summary = df.groupby('zone')[['air_temperature_c', 'relative_humidity_pct', 'soil_moisture_pct', 'co2_ppm', 'gateway_latency_ms', 'gateway_rssi_dbm']].mean().round(2)
zone_summary['water_stress_count'] = df.groupby('zone')['water_stress'].sum()
zone_summary['severe_water_stress_count'] = df.groupby('zone')['severe_water_stress'].sum()
zone_summary['device_health_risk_count'] = df.groupby('zone')['device_health_risk'].sum()
zone_summary['calibration_drift_count'] = df.groupby('zone')['calibration_drift_risk'].sum()
zone_summary['actuator_mismatch_count'] = df.groupby('zone')['actuator_mismatch'].sum()
zone_summary['edge_action_priority_count'] = df.groupby('zone')['edge_action_priority'].sum()
zone_summary.to_csv(outputs_dir / 'greenhouse_zone_summary.csv')

# 3. Output Chart 1: Air temperature over time by zone with threshold
fig, ax = plt.subplots(figsize=(10, 5))
for zone, zone_df in df.groupby('zone'):
    ax.plot(zone_df['timestamp'], zone_df['air_temperature_c'], label=zone)
ax.axhline(thresholds['temperature_high_c'], color='red', linestyle='--', alpha=0.7, label='High temp threshold (27 C)')
ax.set_title('Air temperature by greenhouse zone')
ax.set_ylabel('Air temperature (C)')
ax.set_xlabel('Timestamp')
ax.tick_params(axis='x', rotation=30)
ax.legend()
fig.tight_layout()
fig.savefig(outputs_dir / 'temperature_by_zone.png', dpi=150)
plt.close(fig)

# 4. Output Chart 2: Average gateway latency over time with threshold
latency_by_time = df.groupby('timestamp')['gateway_latency_ms'].mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(latency_by_time.index, latency_by_time.values, color='black', label='Avg Latency (ms)')
ax.axhline(thresholds['latency_high_ms'], color='red', linestyle='--', alpha=0.7, label='High latency threshold (140 ms)')
ax.set_title('Average gateway latency')
ax.set_ylabel('Latency (ms)')
ax.set_xlabel('Timestamp')
ax.tick_params(axis='x', rotation=30)
ax.legend()
fig.tight_layout()
fig.savefig(outputs_dir / 'gateway_latency_over_time.png', dpi=150)
plt.close(fig)

# 5. COMPLETED TODO - Output Chart 3: Soil moisture vs. irrigation actuator behaviour by zone
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for idx, (zone, zone_df) in enumerate(df.groupby('zone')):
    ax = axes[idx]
    ax.plot(zone_df['timestamp'], zone_df['soil_moisture_pct'], color='tab:blue', label=f'{zone} Soil Moisture (%)')
    ax.axhline(thresholds['soil_low_pct'], color='orange', linestyle='--', label='Water Stress Threshold (42%)')
    ax.axhline(thresholds['soil_critical_pct'], color='red', linestyle=':', label='Severe Water Stress (35%)')
    
    irrigating = zone_df[zone_df['irrigation_active'] == 1]
    ax.scatter(irrigating['timestamp'], irrigating['soil_moisture_pct'], color='green', zorder=5, s=30, label='Irrigation Active (1)')
    
    ax.set_title(f'Soil Moisture and Irrigation Actuator Behaviour ({zone})')
    ax.set_ylabel('Soil Moisture (%)')
    ax.legend(loc='upper right', fontsize=8)

axes[1].set_xlabel('Timestamp')
axes[1].tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(outputs_dir / 'soil_moisture_irrigation_behaviour.png', dpi=150)
plt.close(fig)

print("--- ZONE SUMMARY TABLE ---")
print(zone_summary.to_string())

print("\n--- NON-NORMAL ANOMALY WINDOWS ---")
print(df[df['anomaly_label'] != 'normal'][['timestamp', 'zone', 'air_temperature_c', 'soil_moisture_pct', 'co2_ppm', 'fan_active', 'irrigation_active', 'gateway_latency_ms', 'packet_loss_pct', 'anomaly_label']].to_string())

print(f"\nTotal risk_table rows saved: {len(risk_table)}")



--- ZONE SUMMARY TABLE ---
           air_temperature_c  relative_humidity_pct  soil_moisture_pct  co2_ppm  gateway_latency_ms  gateway_rssi_dbm  water_stress_count  severe_water_stress_count  device_health_risk_count  calibration_drift_count  actuator_mismatch_count  edge_action_priority_count
zone                                                                                                                                                                                                                                                                        
north-bed              23.35                  53.90              35.60   482.85               86.95            -64.58                 157                         74                         6                        0                      142                           9
south-bed              22.77                  56.33              38.53   485.70               94.95            -69.58                 129                         42  

## 5. Interpret

Write brief notebook notes here, then develop the full explanation in your Brightspace report.

- What operational risks are visible?
- What data-management requirements does the scenario imply?
- What hardware/platform choices are suitable?
- What privacy and security risks should be managed?

- **Visible Operational Risks:** In the `greenhouse-d` (Water-constrained greenhouse) dataset, `north-bed` experiences acute drought stress (mean soil moisture 35.60%, 157 hours of water stress `< 42%`, 74 hours of severe water stress `< 35%`, and a 7-hour `irrigation_gap` on 24–25 September where irrigation fails to activate), alongside 15 hours of thermal risk (`> 27°C`) and 142 hours of actuator mismatch. Conversely, `south-bed` exhibits critical hardware, ventilation, and telemetry reliability faults, including 48 hours of device health risk (weak RSSI down to `-82 dBm` and 3 consecutive `soil_moisture_pct` sensor dropouts on 22 September 20:00–22:00), 14 night-time negative `light_lux` readings (`-2.40` to `-22.33 lux`) indicating unreported zero-point calibration drift while `sensor_status` reports `ok`, a 7-hour `ventilation_fault` on 24–25 September (`co2_ppm` peaking at 585.17 ppm and temperature reaching 27.80°C), and 36 hours of `edge_action_priority`. Both zones experience a 7-hour `gateway_degraded` window on 26–27 September (22:00–04:00) with latency spiking to 191 ms and packet loss reaching 4.54%.
- **Data-Management Requirements:** Sensor data must undergo automated edge validation (range checks flagging negative `light_lux` and missing dropout values before control logic executes), local time-series buffering (store-and-forward queuing on the gateway during the 14-row `gateway_degraded` window to prevent telemetry loss), and hybrid processing where time-critical closed-loop irrigation and ventilation rules execute locally at the edge/fog tier while aggregated telemetry is synced via MQTT/TLS to a cloud time-series database (e.g., InfluxDB/Grafana) for long-term analytics.
- **Hardware/Platform Suitability:** Low-power microcontroller sensor nodes (such as Waspmote Agricultural PRO boards or industrial ESP32/Arduino nodes with weatherproof enclosures) are best suited for distributed in-bed soil moisture, light, temperature, and CO2 sensing, paired with a Raspberry Pi 4/5 or industrial edge gateway running local edge-control services, MQTT brokering, and store-and-forward buffering to ensure autonomous actuation when cloud connectivity degrades.
- **Security and Privacy Risks:** Key risks include physical tampering or water ingress at sensor nodes, wireless eavesdropping or spoofing of unencrypted sensor payloads, false data injection or calibration drift corrupting automated irrigation decisions, denial-of-service or network congestion at the gateway during `gateway_degraded` windows, and unauthorized access to cloud dashboards exposing proprietary operational schedules. Controls include mutual TLS (mTLS), cryptographic device identity, edge anomaly sanity bounds, physical enclosures, network segmentation, and role-based access control (RBAC).
